### 读取新项目从中间帧检测到的所有人脸框

In [17]:
import os, pickle
import numpy as np
import pandas as pd
  
TV_name = 'the big bang theory' # 'I love my family', 'the big bang theory'

# 从当前TV series的所有中间帧人脸embedding文件中，提取 audio_seg_id, face_idx, bbox 信息，合并成一个大文件，方便后续处理
embs_folder = os.path.join('/data/home/scv7387/run/tv_series_plus/3D-Speaker/egs/3dspeaker/speaker-diarization/runs', TV_name, 'exp_video/embs_video')
midframe_files = [f for f in os.listdir(embs_folder) if f.endswith('_midframe.pkl')]
midframe_files.sort(key=lambda x: int(x.split('_')[0][1:].lstrip('0')))
print(midframe_files)

audio_seg_ids = np.array([], dtype='<U50')
face_idxs = np.array([], dtype=np.int64)
bboxes = np.array([], dtype=np.int64)


for file_idx, midframe_file in enumerate(midframe_files):
    visual_embs_file = os.path.join(embs_folder, midframe_file)
    with open(visual_embs_file, 'rb') as f:
        stat_obj = pickle.load(f)
        if file_idx == 0:
            audio_seg_ids = stat_obj['audio_seg_id'] # np.ndarray, (N, )
            face_idxs = stat_obj['face_idx'] # np.ndarray, (N, )
            bboxes = stat_obj['bbox'] # np.ndarray, (N, 4)
        else:
            audio_seg_ids = np.hstack((audio_seg_ids, stat_obj['audio_seg_id']))
            face_idxs = np.hstack((face_idxs, stat_obj['face_idx']))
            bboxes = np.vstack((bboxes, stat_obj['bbox']))

print(audio_seg_ids.shape)
print(face_idxs.shape)
print(bboxes.shape)

print(audio_seg_ids[:10])
print(face_idxs[:10])
print(bboxes[:10, :])

['E01_midframe.pkl', 'E02_midframe.pkl', 'E03_midframe.pkl', 'E04_midframe.pkl', 'E05_midframe.pkl', 'E06_midframe.pkl', 'E07_midframe.pkl', 'E08_midframe.pkl', 'E09_midframe.pkl', 'E10_midframe.pkl', 'E11_midframe.pkl', 'E12_midframe.pkl', 'E13_midframe.pkl', 'E14_midframe.pkl', 'E15_midframe.pkl', 'E16_midframe.pkl', 'E17_midframe.pkl']
(14646,)
(14646,)
(14646, 4)
['E01-1' 'E01-1' 'E01-2' 'E01-2' 'E01-3' 'E01-3' 'E01-4' 'E01-4' 'E01-5'
 'E01-5']
[0 1 0 1 0 1 0 1 0 1]
[[ 985  186 1038  258]
 [1039  257 1095  329]
 [ 924  182  985  267]
 [1049  267 1113  346]
 [1108  225 1184  322]
 [ 879  149  960  257]
 [1128  255 1212  367]
 [ 803  141  887  270]
 [ 921  208 1144  516]
 [1803  449 1920  783]]


### 筛选需要标注的人脸框

In [18]:
# 读取旧项目下标注人脸归属的xlsx文件（已经复制到当前项目中）
anno_xlsx = os.path.join('/data/home/scv7387/run/tv_series_plus/dataset', TV_name, 'annotation', 'faces_annotation_with_loc.xlsx')
df = pd.read_excel(anno_xlsx)
df['audio_seg_id'] = df.apply(lambda row: f"E{int(row['Episode']):02d}-{int(row['Frame Index'])}", axis=1)
audio_seg_id_set = set(df['audio_seg_id'].unique()) # 获取所有unique的audio_seg_id

# 筛选在原来项目中标注过人脸的部分台词片段
mask = np.array([seg_id in audio_seg_id_set for seg_id in audio_seg_ids])
audio_seg_ids = audio_seg_ids[mask]
face_idxs = face_idxs[mask]
bboxes = bboxes[mask]

print(audio_seg_ids.shape, face_idxs.shape, bboxes.shape)
print(audio_seg_ids[:10])
print(face_idxs[:10])
print(bboxes[:10, :])

(1069,) (1069,) (1069, 4)
['E01-4' 'E01-4' 'E01-32' 'E01-32' 'E01-50' 'E01-50' 'E01-104' 'E01-104'
 'E01-126' 'E01-126']
[0 1 0 1 0 1 0 1 0 1]
[[1128  255 1212  367]
 [ 803  141  887  270]
 [ 566  201  797  520]
 [1080  161 1295  446]
 [ 987  136 1085  257]
 [1073  223 1172  353]
 [1207  180 1389  412]
 [ 849  347 1023  575]
 [ 846  209 1084  534]
 [ 972  761 1020  820]]


### 根据旧项目的标注结果，完成新项目中大部分人脸框的自动标注

In [19]:
import pandas as pd
import numpy as np

# IOU Function to calculate overlap between two boxes
def bb_intersection_over_union(boxA, boxB, evalCol=False):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    if evalCol == True:
        iou = interArea / float(boxAArea)
    else:
        iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

# 1. 初始化新的DataFrame
df_new = pd.DataFrame(columns=list(df.columns) + ['x1_old', 'y1_old', 'x2_old', 'y2_old'])
# 2. 逐行处理
for seg_id, face_idx, bbox in zip(audio_seg_ids, face_idxs, bboxes):
    row = {col: '' for col in df_new.columns}
    row['audio_seg_id'] = seg_id
    row['face index'] = face_idx
    row['x1'], row['y1'], row['x2'], row['y2'] = bbox.tolist()
    # 2.1 筛选同audio_seg_id且bbox合法的行
    df_match = df[(df['audio_seg_id'] == seg_id) & (df[['x1', 'y1', 'x2', 'y2']] != -1).all(axis=1)]


    # 2.2 计算IOU，判断IOU条件
    iou_list = []
    for _, match_row in df_match.iterrows():
        boxA = bbox
        boxB = np.array([match_row['x1'], match_row['y1'], match_row['x2'], match_row['y2']])
        
        iou = bb_intersection_over_union(boxA, boxB)
        iou_list.append(iou)

    if len(iou_list) > 0:
        max_iou = max(iou_list)
        idx_max = np.argmax(iou_list)
        if max_iou > 0.75 and all(i < 0.5 or i == max_iou for i in iou_list):
            match_row = df_match.iloc[idx_max]
            row['face label'] = match_row['face label']
            row['x1_old'], row['y1_old'], row['x2_old'], row['y2_old'] = match_row['x1'], match_row['y1'], match_row['x2'], match_row['y2']

    # 2.3 获取同audio_seg_id的所有'all faces in frame'集合
    all_faces_set = set(df[df['audio_seg_id'] == seg_id]['all faces in frame'].dropna().unique())
    if len(all_faces_set) == 1:
        row['all faces in frame'] = list(all_faces_set)[0]    
    
    # 2.4 解析Episode和Frame Index
    try:
        ep, fr = seg_id.split('-')
        row['Episode'] = int(ep[1:])
        row['Frame Index'] = int(fr)
    except:
        pass
    df_new = pd.concat([df_new, pd.DataFrame([row])], ignore_index=True)

# 3. 检查一个关键帧中多个facebox匹配到了旧项目中同一个bbox的情况，清空这些facebox的label
for seg_id in df_new['audio_seg_id'].unique():
    sub_df = df_new[df_new['audio_seg_id'] == seg_id]
    bbox_cols = ['x1', 'y1', 'x2', 'y2']
    duplicated = sub_df.duplicated(subset=bbox_cols, keep=False)
    if duplicated.any():
        idxs = sub_df.index[duplicated]
        df_new.loc[idxs, ['face label', 'all faces in frame']] = ''

# 4. 保存
df_new.to_excel(anno_xlsx.replace('.xlsx', '_new.xlsx'), index=False)

### 将未标注的人脸框导出图像，以便于手动标注

In [ ]:
import cv2, os


TV_name = 'the big bang theory' # 'I love my family', 'the big bang theory'
anno_xlsx_new = os.path.join('/data/home/scv7387/run/tv_series_plus/dataset', TV_name, 'annotation', 'faces_annotation_with_loc_new.xlsx')
frame_folder = os.path.join('/data/home/scv7387/run/TV_series/dataset', TV_name, 'triples/speaker_image')
frame_folder_save = os.path.join('/data/home/scv7387/run/tv_series_plus/dataset', TV_name, 'middle_frame2annotate')
os.makedirs(frame_folder_save, exist_ok=True)

# 读取df_new，筛选出'face label'为空的行
df_new = pd.read_excel(anno_xlsx_new)
df_unlabeled = df_new[df_new['face label'] == '']
# 获取所有audio_seg_id的集合
audio_seg_ids_unlabeled = set(df_unlabeled['audio_seg_id'].unique())


for seg_id in audio_seg_ids_unlabeled:
    img_path = os.path.join(frame_folder, f'{seg_id}.jpg')
    img = cv2.imread(img_path)
    if img is None:
        print(f'Image not found: {img_path}')
        continue

    rows = df_unlabeled[df_unlabeled['audio_seg_id'] == seg_id]
    for _, row in rows.iterrows():
        x1, y1, x2, y2 = int(row['x1']), int(row['y1']), int(row['x2']), int(row['y2'])
        face_idx = str(row['face index'])
        # 画红色方框
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
        # 标注face index
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        cv2.putText(img, face_idx, (cx, cy), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)

    save_path = os.path.join(frame_folder_save, f'{seg_id}.jpg')
    cv2.imwrite(save_path, img)
